# MOD-03 · NB-03 — Survival Analysis (Kaplan-Meier & Log-Rank)
### Health Analytics with Python · Module 03: Statistical Inference
---
**Learning objectives**
- Understand censoring and the survival analysis framework
- Estimate and plot Kaplan-Meier survival curves with confidence bands
- Compare survival curves with the log-rank test and Wilcoxon test
- Compute and interpret median survival time and restricted mean survival
- Stratify survival by clinical subgroups and annotate at-risk tables

**Estimated time:** 2 hours | **Level:** Intermediate | **Libraries:** `pandas`, `numpy`, `lifelines`, `matplotlib`


## 1. Setup & Survival Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os; os.makedirs('/tmp/mod03',exist_ok=True)

try:
    from lifelines import KaplanMeierFitter, logrank_test
    from lifelines.statistics import multivariate_logrank_test
    from lifelines.utils import median_survival_times
    print("lifelines imported successfully.")
except ImportError:
    print("Run: pip install lifelines")

plt.rcParams.update({'figure.dpi':120,'figure.facecolor':'white',
                     'axes.spines.top':False,'axes.spines.right':False})

# ── Synthetic survival dataset: time to 30-day readmission ───────────────────
np.random.seed(42); N = 1500
def logistic(x): return 1/(1+np.exp(-x))
base = np.random.normal(size=(N,3))
np.random.seed(99)

diabetes      = np.random.binomial(1,logistic(0.6*base[:,0]-0.5),N)
heart_failure = np.random.binomial(1,logistic(0.4*base[:,1]-1.2),N)
copd          = np.random.binomial(1,logistic(0.5*base[:,2]-1.0),N)
payers        = np.random.choice(['Medicare','Medicaid','Private','Self-pay'],N,p=[0.42,0.22,0.28,0.08])
ages          = np.random.normal(62,16,N).clip(18,95).astype(int)
sexes         = np.random.choice(['M','F'],N,p=[0.48,0.52])
los_days      = np.random.gamma(2.5,1.8,N).clip(1,30).astype(int)

# Time to readmission (days after discharge, max follow-up = 365)
# Shorter time for patients with HF, diabetes, high LOS
hazard = np.exp(-3.5 + 0.5*heart_failure + 0.3*diabetes + 0.15*(los_days>7) + 0.2*copd)
time_to_event = np.random.exponential(1/hazard).clip(1, 365).round(0).astype(int)

# Censoring: some patients lost to follow-up before 365 days (30% censored)
censor_time   = np.random.uniform(30, 365, N).round(0).astype(int)
observed      = (time_to_event <= censor_time).astype(int)   # 1 = event observed
duration      = np.where(observed==1, time_to_event, censor_time)

df = pd.DataFrame({
    'patient_id'   : [f'PT-{i:05d}' for i in range(1,N+1)],
    'age':ages,'sex':sexes,'payer':payers,'los_days':los_days,
    'diabetes':diabetes,'heart_failure':heart_failure,'copd':copd,
    'duration_days': duration,
    'event_observed': observed,
})
df['age_group'] = pd.cut(df['age'],[17,44,64,74,95],labels=['18-44','45-64','65-74','75+'])
df['high_risk']  = ((df['heart_failure']==1)|(df['diabetes']==1)).astype(int)

print(f"Survival dataset: {df.shape}")
print(f"Event rate (readmission observed): {df.event_observed.mean()*100:.1f}%")
print(f"Censoring rate               : {(1-df.event_observed.mean())*100:.1f}%")
print(f"Median follow-up time        : {df.duration_days.median():.0f} days")
df[['duration_days','event_observed']].describe().round(1)


## 2. The Censoring Concept

> **Censoring** occurs when the event has not yet happened by the end of follow-up, or when a patient is lost. The censored observation still contributes information: we know the patient was event-free up to their last contact.

| Censoring type | Meaning | Example |
|---|---|---|
| Right-censored | Event not yet occurred at study end | Patient still alive at end of follow-up |
| Left-censored | Event occurred before first observation | Readmission before first clinic visit |
| Interval-censored | Event occurred between two observations | Only know readmission was between visits |

In most clinical registry analyses, **right-censoring** is the norm.


In [ ]:
# Visualise the censoring structure for a small sample
sample = df.sample(25,random_state=0).sort_values('duration_days')
fig,ax = plt.subplots(figsize=(10,7))

for i,(_, row) in enumerate(sample.iterrows()):
    color  = '#D65F5F' if row.event_observed==1 else '#4878CF'
    marker = 'x' if row.event_observed==1 else 'o'
    ax.hlines(i,0,row.duration_days,color=color,lw=1.5,alpha=0.7)
    ax.plot(row.duration_days,i,marker=marker,color=color,ms=8,
            markeredgewidth=2 if marker=='x' else 1)

event_patch  = plt.Line2D([0],[0],color='#D65F5F',lw=2,marker='x',ms=8,markeredgewidth=2,label='Event (readmission)')
censor_patch = plt.Line2D([0],[0],color='#4878CF',lw=2,marker='o',ms=8,label='Censored')
ax.legend(handles=[event_patch,censor_patch],fontsize=10,loc='lower right')
ax.set_xlabel('Days post-discharge'); ax.set_ylabel('Patient (sample of 25)')
ax.set_title('Patient follow-up timelines — events vs censored observations')
plt.tight_layout()
plt.savefig('/tmp/mod03/censoring_diagram.png',bbox_inches='tight'); plt.show()


## 3. Kaplan-Meier Estimator — Overall Cohort

In [ ]:
kmf = KaplanMeierFitter()
kmf.fit(df['duration_days'], event_observed=df['event_observed'],
        label='All patients')

fig,axes = plt.subplots(1,2,figsize=(14,5))

# Panel A — KM curve with 95% CI
ax = axes[0]
kmf.plot_survival_function(ax=ax, ci_show=True, color='#1F4E79',
                            ci_alpha=0.15, linewidth=2)
ax.axhline(0.5,color='red',ls=':',lw=1.2,label='S(t)=0.50 (median)')
med = kmf.median_survival_time_
ax.axvline(med,color='red',ls=':',lw=1.2)
ax.text(med+5,0.52,f'Median = {med:.0f} days',fontsize=9,color='red')
ax.set_xlabel('Days post-discharge')
ax.set_ylabel('Readmission-free probability S(t)')
ax.set_title('Kaplan-Meier survival curve — overall cohort')
ax.legend(fontsize=9)
ax.set_ylim(0,1.05)

# Panel B — Cumulative incidence (1 - S(t))
ax = axes[1]
kmf.plot_cumulative_density(ax=ax, color='#D65F5F', linewidth=2, ci_show=True, ci_alpha=0.15)
ax.set_xlabel('Days post-discharge')
ax.set_ylabel('Cumulative readmission probability F(t)')
ax.set_title('Cumulative incidence of readmission')
ax.set_ylim(0,1.05)

plt.tight_layout()
plt.savefig('/tmp/mod03/km_overall.png',bbox_inches='tight'); plt.show()

# Survival probabilities at key time points
print("Survival probabilities:")
for t in [30,60,90,180,365]:
    prob = kmf.predict(t)
    print(f"  S({t:3d} days) = {prob:.3f}  → {(1-prob)*100:.1f}% readmitted by day {t}")
print(f"\nMedian survival time: {med:.0f} days (50% readmitted by this day)")


## 4. Stratified KM Curves & Log-Rank Test

In [ ]:
fig,axes = plt.subplots(1,2,figsize=(14,5))
colors_hf = {'0 - No HF':'#4878CF','1 - Heart failure':'#D65F5F'}
colors_dm = {'0 - No DM':'#6ACC65','1 - Diabetes':'#B47CC7'}

for (ax, col, labels_map, title, test_label) in [
    (axes[0], 'heart_failure', {0:'0 - No HF',1:'1 - Heart failure'}, 
     'Time to readmission by Heart Failure status','Heart failure'),
    (axes[1], 'diabetes',      {0:'0 - No DM',1:'1 - Diabetes'},
     'Time to readmission by Diabetes status','Diabetes'),
]:
    colors_used = list(colors_hf.values()) if 'HF' in title else list(colors_dm.values())
    kmfs = {}
    for val, label in labels_map.items():
        mask = df[col]==val
        kmf_ = KaplanMeierFitter()
        kmf_.fit(df.loc[mask,'duration_days'],
                 event_observed=df.loc[mask,'event_observed'],
                 label=label)
        kmf_.plot_survival_function(ax=ax, ci_show=True, ci_alpha=0.12,
                                     color=colors_used[val], linewidth=2)
        kmfs[val] = kmf_

    # Log-rank test
    g0 = df[df[col]==0]; g1 = df[df[col]==1]
    lr = logrank_test(g0['duration_days'], g1['duration_days'],
                      event_observed_A=g0['event_observed'],
                      event_observed_B=g1['event_observed'])
    ax.text(0.05,0.15,f'Log-rank p = {lr.p_value:.4f}',
            transform=ax.transAxes,fontsize=10,
            bbox=dict(facecolor='white',alpha=0.8,edgecolor='gray',pad=3))
    ax.set_xlabel('Days post-discharge'); ax.set_ylabel('Readmission-free S(t)')
    ax.set_title(title); ax.legend(fontsize=9); ax.set_ylim(0,1.05)

plt.tight_layout()
plt.savefig('/tmp/mod03/km_stratified.png',bbox_inches='tight'); plt.show()


## 5. Multi-Group Comparison (>2 Groups)

In [ ]:
fig,ax = plt.subplots(figsize=(10,6))
payer_colors = {'Medicare':'#1F4E79','Medicaid':'#D65F5F',
                'Private':'#6ACC65','Self-pay':'#B47CC7'}
payer_groups = []
payer_labels = []
payer_events = []

for payer, color in payer_colors.items():
    mask = df['payer']==payer
    sub  = df[mask]
    kmf_ = KaplanMeierFitter()
    kmf_.fit(sub['duration_days'], event_observed=sub['event_observed'], label=payer)
    kmf_.plot_survival_function(ax=ax, ci_show=False, color=color, linewidth=2)
    payer_groups.append(sub['duration_days'].values)
    payer_labels.append(np.full(len(sub), payer))
    payer_events.append(sub['event_observed'].values)

# Multivariate log-rank test
all_dur   = np.concatenate(payer_groups)
all_grp   = np.concatenate(payer_labels)
all_evt   = np.concatenate(payer_events)
mlr = multivariate_logrank_test(all_dur, all_grp, all_evt)

ax.text(0.05,0.15,f'Multi-group log-rank p = {mlr.p_value:.4f}',
        transform=ax.transAxes,fontsize=11,
        bbox=dict(facecolor='white',alpha=0.85,edgecolor='gray',pad=4))
ax.set_xlabel('Days post-discharge'); ax.set_ylabel('Readmission-free probability S(t)')
ax.set_title('Kaplan-Meier curves by payer type'); ax.set_ylim(0,1.05)
ax.legend(title='Payer',fontsize=9)
plt.tight_layout()
plt.savefig('/tmp/mod03/km_payer.png',bbox_inches='tight'); plt.show()

# Median survival by payer
print("Median readmission-free survival by payer:")
for payer in payer_colors:
    mask = df['payer']==payer
    kmf_.fit(df.loc[mask,'duration_days'],df.loc[mask,'event_observed'])
    print(f"  {payer:12s}: {kmf_.median_survival_time_:.0f} days")


## 6. At-Risk Table & Restricted Mean Survival Time

In [ ]:
# At-risk table — critical for honest reporting of KM curves
kmf_hf1 = KaplanMeierFitter()
kmf_hf0 = KaplanMeierFitter()
mask1 = df['heart_failure']==1; mask0 = df['heart_failure']==0
kmf_hf1.fit(df.loc[mask1,'duration_days'],df.loc[mask1,'event_observed'],label='Heart failure')
kmf_hf0.fit(df.loc[mask0,'duration_days'],df.loc[mask0,'event_observed'],label='No heart failure')

fig,ax = plt.subplots(figsize=(11,5))
kmf_hf1.plot_survival_function(ax=ax,color='#D65F5F',ci_show=True,ci_alpha=0.12,lw=2)
kmf_hf0.plot_survival_function(ax=ax,color='#4878CF',ci_show=True,ci_alpha=0.12,lw=2)

# Add at-risk counts at time points
time_points = [0,30,60,90,180,270,365]
for y_offset, kmf_, color, label in [
        (-0.08,kmf_hf1,'#D65F5F','HF=Yes'),
        (-0.14,kmf_hf0,'#4878CF','HF=No')]:
    at_risk = [kmf_.durations[kmf_.durations>=t].shape[0] for t in time_points]
    for t,n in zip(time_points,at_risk):
        ax.text(t,y_offset,str(n),ha='center',fontsize=8,color=color,
                transform=ax.get_xaxis_transform())

ax.text(-15,-0.08,'At risk:',ha='left',fontsize=8,transform=ax.get_xaxis_transform())
ax.set_xlabel('Days post-discharge (at-risk counts below axis)')
ax.set_ylabel('Readmission-free probability S(t)')
ax.set_title('KM curves with at-risk table — Heart failure subgroups')
ax.legend(fontsize=9); ax.set_ylim(-0.20,1.05)
plt.tight_layout()
plt.savefig('/tmp/mod03/km_atrisk.png',bbox_inches='tight'); plt.show()

# Restricted mean survival time (RMST) up to t* = 180 days
from lifelines.utils import restricted_mean_survival_time
rmst_hf1 = restricted_mean_survival_time(kmf_hf1, t=180)
rmst_hf0 = restricted_mean_survival_time(kmf_hf0, t=180)
print(f"Restricted Mean Survival Time (RMST) up to 180 days:")
print(f"  Heart failure : {rmst_hf1:.1f} days readmission-free")
print(f"  No HF         : {rmst_hf0:.1f} days readmission-free")
print(f"  RMST diff     : {rmst_hf0-rmst_hf1:.1f} days advantage for no-HF group")


## 7. Interpreting Crossing Survival Curves

In [ ]:
# Simulate crossing curves — when log-rank may be misleading
np.random.seed(33)
n_cross = 500
# Group A: early-risk high, late-risk low
t_A = np.random.exponential(60, n_cross).clip(1,365).round(0)
e_A = np.random.binomial(1, 0.65, n_cross)
# Group B: early-risk low, late-risk high (crossing pattern)
t_B = np.concatenate([
    np.random.exponential(200, n_cross//2).clip(1,180),  # early survivors
    np.random.exponential(40,  n_cross//2).clip(181,365) # late events
]).round(0)
e_B = np.random.binomial(1,0.60,n_cross)

kmf_A = KaplanMeierFitter(); kmf_A.fit(t_A,e_A,label='Group A (early risk)')
kmf_B = KaplanMeierFitter(); kmf_B.fit(t_B,e_B,label='Group B (late risk)')
lr_cross = logrank_test(t_A,t_B,event_observed_A=e_A,event_observed_B=e_B)

fig,ax = plt.subplots(figsize=(9,5))
kmf_A.plot_survival_function(ax=ax,color='#D65F5F',lw=2,ci_show=True,ci_alpha=0.1)
kmf_B.plot_survival_function(ax=ax,color='#4878CF',lw=2,ci_show=True,ci_alpha=0.1)
ax.text(0.05,0.15,f'Log-rank p = {lr_cross.p_value:.4f}
(may miss crossing)',
        transform=ax.transAxes,fontsize=10,
        bbox=dict(facecolor='white',alpha=0.85,edgecolor='gray',pad=3))
ax.set_xlabel('Days'); ax.set_ylabel('S(t)')
ax.set_title('Crossing survival curves — log-rank test loses power
Consider Wilcoxon (Breslow) test or weighted log-rank')
ax.legend(fontsize=9); ax.set_ylim(0,1.05)
plt.tight_layout()
plt.savefig('/tmp/mod03/km_crossing.png',bbox_inches='tight'); plt.show()


## 8. Knowledge Check
1. A patient who withdraws from the study on day 45 without experiencing the event — how should they be coded (duration, event_observed)?
2. The log-rank p-value for HF vs no-HF is <0.001. Is this sufficient to conclude HF causes faster readmission? What is missing?
3. RMST for HF group = 94 days vs no-HF = 131 days. How would you phrase this for a clinical report?
4. When would you prefer Wilcoxon (Breslow) over the log-rank test?
5. Replicate the payer analysis restricted to patients aged 65+ only. Does the ranking of median survival times change?


In [ ]:
# Exercise 5 — KM by payer for 65+ only
df_65 = df[df['age']>=65]
fig,ax = plt.subplots(figsize=(9,5))
for payer,color in [('Medicare','#1F4E79'),('Medicaid','#D65F5F'),('Private','#6ACC65')]:
    mask = df_65['payer']==payer
    if mask.sum() < 20: continue
    kmf_ = KaplanMeierFitter()
    kmf_.fit(df_65.loc[mask,'duration_days'],df_65.loc[mask,'event_observed'],label=payer)
    kmf_.plot_survival_function(ax=ax,color=color,lw=2,ci_show=False)
    print(f"  {payer:12s} (65+): median = {kmf_.median_survival_time_:.0f} days, N={mask.sum()}")
ax.set_xlabel('Days'); ax.set_ylabel('S(t)'); ax.legend()
ax.set_title('KM by payer — patients aged 65+ only')
plt.tight_layout(); plt.show()


---
**Next → NB-04: Regression Modelling — Linear Regression for Health Outcomes**